In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_csv("../orders_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (6000, 13)


,order_id,product_category,price_inr,discount_pct,payment_method,customer_tenure_days,num_previous_orders,num_previous_returns,delivery_distance_km,delivery_days,is_weekend_order,rating_given,returned
0,1,Footwear,2572.0,23.8,Prepaid_Card,17,3,0,604.6,1,0,2.0,0
1,2,Electronics,16689.0,6.7,COD,104,3,0,166.4,9,0,2.0,0
2,3,Footwear,800.0,51.3,Prepaid_Card,103,3,0,418.8,8,0,3.0,0
3,4,Home,7930.0,49.7,COD,1479,33,4,143.1,5,0,3.0,1
4,5,Apparel,715.0,0.0,COD,95,0,0,335.5,3,1,2.0,0


In [6]:
X = df.drop(columns=["returned"])
y = df["returned"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

X shape: (6000, 12)
y shape: (6000,)

Target distribution:
returned
0    4635
1    1365
Name: count, dtype: int64

Target proportions:
returned
0    0.7725
1    0.2275
Name: proportion, dtype: float64


In [7]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['order_id', 'price_inr', 'discount_pct', 'customer_tenure_days', 'num_previous_orders', 'num_previous_returns', 'delivery_distance_km', 'delivery_days', 'is_weekend_order', 'rating_given']

Categorical features:
['product_category', 'payment_method']


In [8]:
X = X.drop(columns=["order_id"])

print("X shape:", X.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (6000, 11)

Features:
['product_category', 'price_inr', 'discount_pct', 'payment_method', 'customer_tenure_days', 'num_previous_orders', 'num_previous_returns', 'delivery_distance_km', 'delivery_days', 'is_weekend_order', 'rating_given']


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Training set: (4800, 11)
Test set: (1200, 11)

Training target distribution:
returned
0    0.7725
1    0.2275
Name: proportion, dtype: float64

Test target distribution:
returned
0    0.7725
1    0.2275
Name: proportion, dtype: float64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [11]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['price_inr', 'discount_pct', 'customer_tenure_days', 'num_previous_orders', 'num_previous_returns', 'delivery_distance_km', 'delivery_days', 'is_weekend_order', 'rating_given']

Categorical features:
['product_category', 'payment_method']


In [12]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [13]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [15]:
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

In [16]:
print("Preprocessor created successfully.")
print(preprocessor)

Preprocessor created successfully.
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['price_inr', 'discount_pct',
                                  'customer_tenure_days', 'num_previous_orders',
                                  'num_previous_returns',
                                  'delivery_distance_km', 'delivery_days',
                                  'is_weekend_order', 'rating_given']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
   

In [14]:
from sklearn.dummy import DummyClassifier

In [15]:
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

In [22]:
dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score, f1_score

dummy_accuracy = accuracy_score(y_test, y_pred_dummy)
dummy_f1 = f1_score(y_test, y_pred_dummy, pos_label=1)

print("DummyClassifier Accuracy:", round(dummy_accuracy, 4))
print("DummyClassifier F1-score (returned=1):", round(dummy_f1, 4))

DummyClassifier Accuracy: 0.7725
DummyClassifier F1-score (returned=1): 0.0


### DummyClassifier Baseline — Interpretation

The DummyClassifier achieved an accuracy of 77.25%, but its F1-score for the `returned=1` class was 0.0. This happens because the most-frequent strategy predicts every order as `returned=0`, since non-returned orders are the majority class. Although this produces a seemingly high accuracy, the model has zero recall for actual returns and therefore fails to identify any returned orders. This demonstrates the **high accuracy, zero recall** trap: accuracy alone can be misleading when the target classes are imbalanced. Therefore, model performance should be compared against a baseline and evaluated using metrics that are aligned with the real business problem, such as recall, precision, F1-score, and ROC-AUC, rather than relying on accuracy alone.

In [24]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        random_state=42,
        max_iter=1000
    ))
])

In [25]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price_inr', 'discount_pct',
                                                   'customer_tenure_days',
                                                   'num_previous_orders',
                                                   'num_previous_returns',
                                                   'delivery_distance_km',
                                                   'delivery_days',
                                                   'is_weekend_order',
                                                   'rating_given']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['product_category',
                                                   'payment_method'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [26]:
y_pred_log = logistic_model.predict(X_test)
y_proba_log = logistic_model.predict_proba(X_test)[:, 1]

In [27]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)

log_accuracy = accuracy_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log, pos_label=1)
log_recall = recall_score(y_test, y_pred_log, pos_label=1)
log_precision = precision_score(y_test, y_pred_log, pos_label=1)
log_roc_auc = roc_auc_score(y_test, y_proba_log)

print("Logistic Regression — Default Threshold (0.5)")
print("Accuracy :", round(log_accuracy, 4))
print("F1       :", round(log_f1, 4))
print("Recall   :", round(log_recall, 4))
print("Precision:", round(log_precision, 4))
print("ROC-AUC  :", round(log_roc_auc, 4))

Logistic Regression — Default Threshold (0.5)
Accuracy : 0.5917
F1       : 0.3921
Recall   : 0.5788
Precision: 0.2964
ROC-AUC  : 0.6253


In [28]:
import numpy as np

thresholds = np.arange(0.10, 0.901, 0.01)

In [31]:
threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba_log >= threshold).astype(int)
    
    f1 = f1_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    precision = precision_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    
    threshold_results.append({
        "threshold": threshold,
        "f1": f1,
        "recall": recall,
        "precision": precision
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.head()

,threshold,f1,recall,precision
0,0.10,0.370672,1.0,0.2275
1,0.11,0.370672,1.0,0.2275
2,0.12,0.370672,1.0,0.2275
3,0.13,0.370672,1.0,0.2275
4,0.14,0.370672,1.0,0.2275


In [32]:
print("Minimum probability:", y_proba_log.min())
print("Maximum probability:", y_proba_log.max())
print("Mean probability:", y_proba_log.mean())

print("\nProbability percentiles:")
print(np.percentile(y_proba_log, [0, 10, 25, 50, 75, 90, 100]))

Minimum probability: 0.2013711249662326
Maximum probability: 0.8976601857946006
Mean probability: 0.4839872714810472

Probability percentiles:
[0.20137112 0.32149733 0.38643683 0.48095446 0.5759084  0.65648326
 0.89766019]


In [33]:
print(threshold_df.to_string(index=False))

 threshold       f1   recall  precision
      0.10 0.370672 1.000000   0.227500
      0.11 0.370672 1.000000   0.227500
      0.12 0.370672 1.000000   0.227500
      0.13 0.370672 1.000000   0.227500
      0.14 0.370672 1.000000   0.227500
      0.15 0.370672 1.000000   0.227500
      0.16 0.370672 1.000000   0.227500
      0.17 0.370672 1.000000   0.227500
      0.18 0.370672 1.000000   0.227500
      0.19 0.370672 1.000000   0.227500
      0.20 0.370672 1.000000   0.227500
      0.21 0.369565 0.996337   0.226856
      0.22 0.369565 0.996337   0.226856
      0.23 0.370068 0.996337   0.227235
      0.24 0.371585 0.996337   0.228380
      0.25 0.370725 0.992674   0.227923
      0.26 0.370014 0.985348   0.227773
      0.27 0.372576 0.985348   0.229718
      0.28 0.375698 0.985348   0.232097
      0.29 0.377652 0.978022   0.234005
      0.30 0.376167 0.959707   0.233929
      0.31 0.373547 0.941392   0.233001
      0.32 0.373708 0.926740   0.234043
      0.33 0.379154 0.919414   0.238820


In [36]:
# Find optimal threshold based on maximum F1 score
best_idx = threshold_df['f1'].idxmax()
best_threshold = threshold_df.loc[best_idx, 'threshold']
best_f1 = threshold_df.loc[best_idx, 'f1']
best_recall = threshold_df.loc[best_idx, 'recall']
best_precision = threshold_df.loc[best_idx, 'precision']

In [37]:
print("Logistic Regression Threshold Optimization")
print("-------------------------------------------")
print("Default threshold:", 0.50)
print("Default recall:", round(log_recall, 4))
print("Default precision:", round(log_precision, 4))
print("Default F1:", round(log_f1, 4))

print("\nF1-maximising threshold:", round(best_threshold, 2))
print("Best F1:", round(best_f1, 4))
print("Recall at best threshold:", round(best_recall, 4))
print("Precision at best threshold:", round(best_precision, 4))

recall_gain = best_recall - log_recall
precision_change = best_precision - log_precision

print("\nRecall improvement:", round(recall_gain, 4))
print("Precision change:", round(precision_change, 4))

Logistic Regression Threshold Optimization
-------------------------------------------
Default threshold: 0.5
Default recall: 0.5788
Default precision: 0.2964
Default F1: 0.3921

F1-maximising threshold: 0.44
Best F1: 0.4091
Recall at best threshold: 0.7582
Precision at best threshold: 0.2801

Recall improvement: 0.1795
Precision change: -0.0163


Business trade-off: Changing the decision threshold from 0.50 to 0.44 makes the model more aggressive in identifying potentially returned orders. This increases recall from 57.88% to 75.82%, meaning the model catches substantially more actual returns. However, precision decreases from 29.64% to 28.01%, so more non-returned orders are incorrectly flagged as potential returns. In business terms, we are accepting more false positives in exchange for reducing false negatives, because failing to identify an order that will actually be returned can be more costly than unnecessarily flagging an order for additional attention.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "price_inr",
    "discount_pct",
    "customer_tenure_days",
    "num_previous_orders",
    "num_previous_returns",
    "delivery_distance_km",
    "delivery_days",
    "is_weekend_order",
    "rating_given"
]

categorical_features = [
    "product_category",
    "payment_method"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor recreated successfully.")

Preprocessor recreated successfully.


In [17]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(4800, 11)
(1200, 11)
(4800,)
(1200,)


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [19]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [20]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [21]:
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [6, 10, None]
}

print(param_grid)

{'classifier__n_estimators': [100, 200], 'classifier__max_depth': [6, 10, None]}


In [22]:
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

print("GridSearchCV created successfully.")

GridSearchCV created successfully.


In [23]:
rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['price_inr',
                                                                          'discount_pct',
                                                                          'customer_tenure_days',
                                                                          'num_previous_orders',
                                                                          'num_previous_returns',
                                                                          'delivery...
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['product_category',
                                                                          'payment_method'])])),
                                       ('classifier',
                                        RandomForestClassifier(class_weight='balanced',
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [6, 10, None],
                         'classifier__n_estimators': [100, 200]},
             scoring='roc_auc', verbose=1)

In [24]:
print("Best parameters:")
print(rf_grid.best_params_)

print("\nBest cross-validated ROC-AUC:")
print(round(rf_grid.best_score_, 4))

Best parameters:
{'classifier__max_depth': 6, 'classifier__n_estimators': 100}

Best cross-validated ROC-AUC:
0.6178


In [25]:
# Evaluate the winning Random Forest on the untouched test set

rf_best = rf_grid.best_estimator_

y_test_proba_rf = rf_best.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score

rf_test_roc_auc = roc_auc_score(y_test, y_test_proba_rf)

print("Random Forest — Held-out Test ROC-AUC")
print(f"ROC-AUC: {rf_test_roc_auc:.4f}")

Random Forest — Held-out Test ROC-AUC
ROC-AUC: 0.6143


### Task 6 — Random Forest Results

The Random Forest was tuned using GridSearchCV with 5-fold StratifiedKFold cross-validation and ROC-AUC as the scoring metric. The best configuration was `n_estimators=100` and `max_depth=6`, with `class_weight="balanced"`.

The best cross-validated ROC-AUC was **0.6178**. When evaluated on the untouched held-out test set, the model achieved a ROC-AUC of **0.6143**. The difference between the cross-validated and test ROC-AUC was only **0.0035**, which is well within the required 0.05 tolerance and provides no evidence of severe overfitting.

In [26]:
# Extract the fitted preprocessing step from the winning Random Forest
fitted_preprocessor = rf_best.named_steps["preprocessor"]

# Get the feature names after imputation and one-hot encoding
feature_names = fitted_preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("\nFeature names:")
print(feature_names)

Number of transformed features: 18

Feature names:
['num__price_inr' 'num__discount_pct' 'num__customer_tenure_days'
 'num__num_previous_orders' 'num__num_previous_returns'
 'num__delivery_distance_km' 'num__delivery_days' 'num__is_weekend_order'
 'num__rating_given' 'cat__product_category_Apparel'
 'cat__product_category_Beauty' 'cat__product_category_Electronics'
 'cat__product_category_Footwear' 'cat__product_category_Home'
 'cat__payment_method_COD' 'cat__payment_method_Prepaid_Card'
 'cat__payment_method_Prepaid_UPI' 'cat__payment_method_Wallet']


In [27]:
# Get feature importances from the winning Random Forest
rf_classifier = rf_best.named_steps["classifier"]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_classifier.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("Top 5 Random Forest Feature Importances:")
print(importance_df.head(5).to_string(index=False))

Top 5 Random Forest Feature Importances:
                  feature  importance
  cat__payment_method_COD    0.166461
           num__price_inr    0.137116
num__customer_tenure_days    0.107431
num__delivery_distance_km    0.097244
        num__discount_pct    0.089011


In [28]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance on the held-out test set
perm_result = permutation_importance(
    rf_best,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Get the feature names from the original X_test columns
original_feature_names = X_test.columns

permutation_df = pd.DataFrame({
    "feature": original_feature_names,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

permutation_df = permutation_df.sort_values(
    "importance_mean",
    ascending=False
).reset_index(drop=True)

print("Permutation Importance — All Original Features:")
print(permutation_df.to_string(index=False))

Permutation Importance — All Original Features:
             feature  importance_mean  importance_std
      payment_method         0.097461        0.010158
           price_inr         0.008015        0.003374
num_previous_returns         0.007110        0.002765
    product_category         0.004619        0.005140
       delivery_days        -0.000416        0.003184
    is_weekend_order        -0.001151        0.000760
 num_previous_orders        -0.001642        0.002206
        rating_given        -0.002549        0.001615
delivery_distance_km        -0.002711        0.002439
        discount_pct        -0.002868        0.002856
customer_tenure_days        -0.005192        0.002208


In [29]:
# Original Random Forest top-5 vs permutation importance

top5_impurity = importance_df.head(5).copy()

# Convert one-hot feature names into their original feature groups
def original_feature_name(feature):
    if feature.startswith("num__"):
        return feature.replace("num__", "")
    elif feature.startswith("cat__"):
        return feature.replace("cat__", "").rsplit("_", 1)[0]
    return feature

top5_impurity["original_feature"] = (
    top5_impurity["feature"].apply(original_feature_name)
)

comparison_df = top5_impurity[
    ["feature", "importance"]
].copy()

comparison_df["original_feature"] = top5_impurity["original_feature"]

comparison_df["permutation_importance"] = comparison_df[
    "original_feature"
].map(
    permutation_df.set_index("feature")["importance_mean"]
)

comparison_df = comparison_df[
    ["feature", "importance", "original_feature", "permutation_importance"]
]

print("Impurity Importance vs Permutation Importance:")
print(comparison_df.to_string(index=False))

Impurity Importance vs Permutation Importance:
                  feature  importance     original_feature  permutation_importance
  cat__payment_method_COD    0.166461       payment_method                0.097461
           num__price_inr    0.137116            price_inr                0.008015
num__customer_tenure_days    0.107431 customer_tenure_days               -0.005192
num__delivery_distance_km    0.097244 delivery_distance_km               -0.002711
        num__discount_pct    0.089011         discount_pct               -0.002868


### Task 7 — Model Explanation and Feature Importance

The winning Random Forest's impurity-based feature importance identified `payment_method_COD`, `price_inr`, `customer_tenure_days`, `delivery_distance_km`, and `discount_pct` as its top five features. These features provide plausible signals of return risk: payment method may reflect different purchasing and return behaviours; price may distinguish higher-value purchases with different return patterns; customer tenure can capture differences in customer behaviour; discount level may be associated with purchase motivation and return behaviour; and delivery distance may appear useful to the model because it can act as a proxy for patterns present in the training data.

However, permutation importance on the held-out test set gives a substantially different picture. `payment_method` remains the strongest feature with a permutation importance of 0.097461, while `price_inr` falls to 0.008015. `customer_tenure_days`, `delivery_distance_km`, and `discount_pct` all fall to negative permutation importance values of -0.005192, -0.002711, and -0.002868 respectively. In contrast, `num_previous_returns`, which was not in the impurity-based top five, has the third-highest permutation importance at 0.007110.

The original top-five feature that loses particularly substantial importance under permutation is `delivery_distance_km`, falling from 0.097244 to -0.002711. This is consistent with the data-generating process because delivery distance was not actually used to generate the `returned` target. The large reduction in the apparent importance of `customer_tenure_days` and `discount_pct` also shows that impurity-based importance can identify apparent patterns that do not translate into a meaningful drop in held-out predictive performance.

Impurity-based Random Forest importance can overrate a noisy continuous feature because tree splits can find apparently useful thresholds among many possible values, giving such high-cardinality continuous variables disproportionate importance even when they do not carry genuine predictive signal.

The top five impurity-based features therefore should not be interpreted as definitive causal drivers of returns. Permutation importance provides a more useful test of whether shuffling a feature actually harms predictive performance on unseen data.

In [30]:
from sklearn.metrics import recall_score, precision_score

# Predictions from the winning Random Forest
y_test_pred_rf = rf_best.predict(X_test)

# Combine test features and predictions for subgroup analysis
test_results = X_test.copy()
test_results["actual_returned"] = y_test.values
test_results["predicted_returned"] = y_test_pred_rf

print("Test predictions created successfully.")
print(test_results.shape)

Test predictions created successfully.
(1200, 13)


In [31]:
# Recall and precision by product category

category_results = []

for category in sorted(test_results["product_category"].unique()):
    subset = test_results[test_results["product_category"] == category]

    recall = recall_score(
        subset["actual_returned"],
        subset["predicted_returned"],
        zero_division=0
    )

    precision = precision_score(
        subset["actual_returned"],
        subset["predicted_returned"],
        zero_division=0
    )

    category_results.append({
        "product_category": category,
        "recall": recall,
        "precision": precision,
        "n": len(subset)
    })

category_metrics = pd.DataFrame(category_results)

print("Performance by Product Category:")
print(category_metrics.to_string(index=False))

Performance by Product Category:
product_category   recall  precision   n
         Apparel 0.530000   0.341935 385
          Beauty 0.612903   0.500000 116
     Electronics 0.326923   0.278689 261
        Footwear 0.500000   0.333333 217
            Home 0.647059   0.224490 221


In [32]:
# Recall and precision by payment method

payment_results = []

for method in sorted(test_results["payment_method"].unique()):
    subset = test_results[test_results["payment_method"] == method]

    recall = recall_score(
        subset["actual_returned"],
        subset["predicted_returned"],
        zero_division=0
    )

    precision = precision_score(
        subset["actual_returned"],
        subset["predicted_returned"],
        zero_division=0
    )

    payment_results.append({
        "payment_method": method,
        "recall": recall,
        "precision": precision,
        "n": len(subset)
    })

payment_metrics = pd.DataFrame(payment_results)

print("Performance by Payment Method:")
print(payment_metrics.to_string(index=False))

Performance by Payment Method:
payment_method   recall  precision   n
           COD 0.877419   0.317016 503
  Prepaid_Card 0.000000   0.000000 283
   Prepaid_UPI 0.041667   0.666667 294
        Wallet 0.047619   0.500000 120


### Task 8 — Subgroup / Root-Cause Analysis

The Random Forest shows substantial differences in performance across product categories and payment methods. By product category, Electronics is the weakest subgroup, with recall of 0.327 and precision of 0.279, compared with stronger recall in Home (0.647) and Beauty (0.613). By payment method, the disparity is much larger: COD has recall of 0.877 and precision of 0.317, whereas Prepaid_Card has recall of 0.000, Prepaid_UPI has recall of 0.042, and Wallet has recall of 0.048.

The clearest weak subgroup is therefore Prepaid_Card orders, for which the model detects none of the actual returns in the held-out test set. A concrete next step would be to introduce payment-method-specific decision thresholds rather than applying one global threshold to every order. In particular, the threshold for prepaid orders could be lowered and tuned on validation data to increase recall for these groups, while monitoring the resulting increase in false positives. This would directly target the observed subgroup failure rather than simply collecting more data without a specific intervention.

In [33]:
print(rf_best)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price_inr', 'discount_pct',
                                                   'customer_tenure_days',
                                                   'num_previous_orders',
                                                   'num_previous_returns',
                                                   'delivery_distance_km',
                                                   'delivery_days',
                                                   'is_weekend_order',
                                               

In [37]:
from sklearn.metrics import f1_score, recall_score, precision_score

In [38]:
# Random Forest probability predictions
rf_test_proba = rf_best.predict_proba(X_test)[:, 1]

# Threshold sweep from 0.10 to 0.90 in steps of 0.01
rf_thresholds = np.arange(0.10, 0.901, 0.01)

rf_threshold_results = []

for threshold in rf_thresholds:
    y_pred_threshold = (rf_test_proba >= threshold).astype(int)

    f1 = f1_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    precision = precision_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    rf_threshold_results.append({
        "threshold": threshold,
        "f1": f1,
        "recall": recall,
        "precision": precision
    })

rf_threshold_df = pd.DataFrame(rf_threshold_results)

# Find the threshold that maximizes F1
best_rf_row = rf_threshold_df.loc[
    rf_threshold_df["f1"].idxmax()
]

t_rf = best_rf_row["threshold"]

print("Random Forest Threshold Sweep")
print()
print(f"F1-maximising threshold (t*_rf): {t_rf:.2f}")
print(f"F1        : {best_rf_row['f1']:.4f}")
print(f"Recall    : {best_rf_row['recall']:.4f}")
print(f"Precision : {best_rf_row['precision']:.4f}")

Random Forest Threshold Sweep

F1-maximising threshold (t*_rf): 0.47
F1        : 0.4030
Recall    : 0.5861
Precision : 0.3071


In [39]:
import joblib
import os

# Make sure the models directory exists
os.makedirs("../models", exist_ok=True)

# Save the complete fitted Random Forest pipeline
model_path = "../models/return_risk_model.pkl"

joblib.dump(rf_best, model_path)

print("Model saved successfully.")
print("Path:", model_path)

Model saved successfully.
Path: ../models/return_risk_model.pkl


In [40]:
import joblib
import os

# Reload the saved model
loaded_model = joblib.load("../models/return_risk_model.pkl")

print("Model loaded successfully.")
print("Model type:", type(loaded_model))
print("Pipeline steps:", loaded_model.named_steps.keys())

# Verify that the saved model can produce probabilities
loaded_proba = loaded_model.predict_proba(X_test)[:, 1]

print("predict_proba() working successfully.")
print("Number of test predictions:", len(loaded_proba))
print("First 5 probabilities:", loaded_proba[:5])

Model loaded successfully.
Model type: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: dict_keys(['preprocessor', 'classifier'])
predict_proba() working successfully.
Number of test predictions: 1200
First 5 probabilities: [0.53196528 0.3502429  0.33568491 0.51502673 0.46377577]


### Task 9 — Final Model Artifact and Random Forest Threshold

The final selected model is the tuned Random Forest pipeline from Task 6, including the preprocessing `ColumnTransformer` and the GridSearchCV-selected `RandomForestClassifier`. The model was saved as `models/return_risk_model.pkl` using `joblib.dump()`.

Before saving the artifact, the threshold sweep was repeated using the Random Forest's own `predict_proba()` output on the held-out test set, rather than reusing the Logistic Regression threshold. The F1-maximising Random Forest threshold was:

**t*_rf = 0.47**

At this threshold:

- F1-score: **0.4030**
- Recall: **0.5861**
- Precision: **0.3071**

The saved artifact was subsequently reloaded successfully with `joblib.load()`. The loaded object is a scikit-learn `Pipeline` containing the preprocessing and Random Forest classifier, and its `predict_proba()` method successfully produced probabilities for all 1,200 held-out test observations.